In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch04-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "04-02-physicochemical-descriptors.py"

從 MoleculeNet ESOL 取甲苯、苯酚、苯甲醚和對甲酚，用分子式與結構圖對讀三個描述符。首次執行需要網路下載資料。輸出的最後一欄為 ESOL 的實測 logS。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Draw, rdMolDescriptors

# ESOL，1,128 筆；Delaney DOI 10.1021/ci034243x。
np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
tasks, splits, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), reload=False)
assert sum(map(len, splits)) == 1128
observed = {str(s).strip(): float(y[0]) for split in splits for s, y in zip(split.ids, split.y)}
examples = {"Toluene": "Cc1ccccc1", "Phenol": "c1ccccc1O",
            "Anisole": "COc1ccccc1", "p-Cresol": "Cc1ccc(O)cc1"}

def describe(mol):
    return Descriptors.MolWt(mol), Crippen.MolLogP(mol), rdMolDescriptors.CalcTPSA(mol)

fig, axes = plt.subplots(1, 4, figsize=(10, 3.7))
for ax, (name, smiles) in zip(axes, examples.items()):
    log_s = observed[smiles]
    mol = Chem.MolFromSmiles(smiles)
    mw, log_p, tpsa = describe(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    print(f"{name:8} {formula:6} {mw:7.3f} {log_p:7.4f} {tpsa:6.2f} {log_s:6.2f}")
    ax.imshow(Draw.MolToImage(mol, size=(280, 210)))
    ax.set_title(name, fontweight="bold", color="#183747")
    ax.axis("off")
    ax.text(0.5, -0.04, f"{formula}\nMW   {mw:.2f}\nclogP   {log_p:.4f}\nTPSA   {tpsa:.2f} Å²",
            ha="center", va="top", transform=ax.transAxes, fontsize=11, linespacing=1.5)
fig.subplots_adjust(top=0.95, bottom=0.32, wspace=0.12)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "04-02-physicochemical-descriptors.png", dpi=160, bbox_inches="tight", pad_inches=0.15)
plt.close(fig)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "04-02-physicochemical-descriptors.png")))